# Surgical Gestures — Pipeline Wrapper

Interactive run-and-visualize notebook for the full EEG-eye bridge + ViT training pipeline.

**Flow:** setup → LOUO splits → Phase 1 (EEG) → Phase 2 (eye) → Phase 3 (RDMs) → Phase 4 (ViT training) → 8-fold LOUO → ablation comparison.

Each phase cell runs the corresponding Python script via subprocess, then the next cell inspects / visualizes its output. Skip any phase whose output already exists in `cache/` or `analysis/`.

**Requirement:** kernel must have the project's `requirements.txt` installed. The setup cell sets `PYTHONPATH=src` for the current process.


## 0. Setup — paths, imports, PYTHONPATH

In [ ]:
import json
import os
import pickle
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().resolve()
if REPO.name == 'pipeline':
    REPO = REPO.parent

SRC = REPO / 'src'
PIPELINE = REPO / 'pipeline'
CACHE = REPO / 'cache' / 'eeg_eye_bridge'
ANALYSIS = REPO / 'analysis'

os.environ['PYTHONPATH'] = str(SRC) + os.pathsep + os.environ.get('PYTHONPATH', '')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PYEXE = sys.executable

def run(cmd, **kw):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    return subprocess.run([str(c) for c in cmd], cwd=str(REPO), env=os.environ, check=True, **kw)

print(f'Repo root:   {REPO}')
print(f'Pipeline:    {PIPELINE}')
print(f'Cache root:  {CACHE}')
print(f'Analysis:    {ANALYSIS}')
print(f'Python:      {PYEXE}')

## 1. Generate LOUO splits

Creates `data/splits/<Task>_splits.json` with 8 leave-one-user-out folds per task. Skip if already present.

In [ ]:
run([PYEXE, PIPELINE / 'generate_splits.py'])

In [ ]:
# Inspect splits: show test-surgeon per fold per task
splits_dir = REPO / 'data' / 'splits'
for splits_file in sorted(splits_dir.glob('*_splits.json')):
    with open(splits_file) as f:
        splits = json.load(f)
    print(f'\n{splits_file.name}')
    for fold, parts in splits.items():
        test_trials = parts.get('test', [])
        surgeons = sorted({t.split('_')[-1][0] for t in test_trials}) if test_trials else []
        print(f'  {fold}: test surgeons={surgeons}  (n_test={len(test_trials)})')

## 2. Phase 1 — EEG export

Loads EDF files from `EEG/EEG/`, applies bandpass + notch filtering, runs sliding-window encoders (baseline CNN + predictive-coding), and writes per-trial pickles + manifest to `cache/eeg_eye_bridge/phase1/`.

Pass `--synthetic_only` to skip EDF loading (useful smoke test).

In [ ]:
USE_SYNTHETIC = False  # set True for a fast smoke run without EDFs
cmd = [PYEXE, PIPELINE / 'phase1_run_export.py', '--data_root', REPO]
if USE_SYNTHETIC:
    cmd.append('--synthetic_only')
run(cmd)

In [ ]:
# Inspect Phase 1 manifest + one sample trial
p1 = CACHE / 'phase1'
manifest_p1 = p1 / 'manifest.json'
if manifest_p1.exists():
    manifest = json.loads(manifest_p1.read_text())
    print(f"Contract: {manifest.get('contract_version')}")
    print(f"Trials:   {len(manifest.get('trials', []))}")
    print(f"First few: {[t.get('trial_id') for t in manifest.get('trials', [])[:5]]}")

sample_pickles = sorted((p1 / 'trials').glob('*.pkl')) if (p1 / 'trials').exists() else []
if sample_pickles:
    with open(sample_pickles[0], 'rb') as f:
        trial = pickle.load(f)
    print(f"\nSample trial: {sample_pickles[0].name}")
    for k, v in trial.items():
        shape = getattr(v, 'shape', None) or (len(v) if hasattr(v, '__len__') else '-')
        print(f'  {k}: shape={shape}, dtype={type(v).__name__}')

    # Plot baseline vs predictive-coding embeddings (first PC of each, over windows)
    base = trial.get('baseline_embeddings')
    pc = trial.get('pc_embeddings')
    if base is not None and pc is not None:
        fig, ax = plt.subplots(1, 2, figsize=(10, 3))
        ax[0].imshow(np.asarray(base).T, aspect='auto', cmap='viridis')
        ax[0].set_title('Baseline embeddings (64-d × windows)')
        ax[1].imshow(np.asarray(pc).T, aspect='auto', cmap='viridis')
        ax[1].set_title('PC embeddings (64-d × windows)')
        for a in ax: a.set_xlabel('window'); a.set_ylabel('dim')
        plt.tight_layout(); plt.show()

## 3. Phase 2 — Eye consistency

Reads Phase 1 cache and eye-tracking CSVs (`Eye/EYE/*.csv`), builds fingerprint vectors (HMM occupancy, saccades, pupil), scores EEG↔eye consistency four ways. Output: `cache/eeg_eye_bridge/phase2/`.

In [ ]:
run([PYEXE, PIPELINE / 'phase2_run_phase2.py', '--repo-root', REPO])

In [ ]:
# Inspect Phase 2: consistency scores distribution
p2 = CACHE / 'phase2'
scores_file = p2 / 'eye_consistency_scores.pkl'
if scores_file.exists():
    with open(scores_file, 'rb') as f:
        scores = pickle.load(f)
    print(f'Per-trial records: {len(scores) if hasattr(scores, "__len__") else scores}')
    # Try common keys and render a quick histogram of the fingerprint-Spearman metric
    if isinstance(scores, dict):
        for k, v in list(scores.items())[:8]:
            print(f'  {k}: {type(v).__name__}')
        fp = scores.get('fingerprint_spearman') or scores.get('fingerprint')
        if fp is not None:
            arr = np.asarray(list(fp.values()) if isinstance(fp, dict) else fp, dtype=float)
            arr = arr[np.isfinite(arr)]
            if arr.size:
                plt.figure(figsize=(6, 3))
                plt.hist(arr, bins=20, color='steelblue', edgecolor='white')
                plt.title('Phase 2: fingerprint-Spearman per trial')
                plt.xlabel('Spearman ρ'); plt.ylabel('count')
                plt.tight_layout(); plt.show()

sel_file = p2 / 'selected_representations.json'
if sel_file.exists():
    print('\nSelected representations:')
    print(json.dumps(json.loads(sel_file.read_text()), indent=2))

## 4. Phase 3 — RDM construction

Builds all candidate K×K RDMs (eye-only, EEG-latent, EEG-PE, joint — at task-family and subskill-family granularities) using 1−Spearman, scores them for split-half stability, and writes `rdm_manifest.json`.

In [ ]:
run([PYEXE, PIPELINE / 'phase3_build_rdms.py', '--cache-root', CACHE])

In [ ]:
# Inspect Phase 3 manifest + visualize every RDM as a heatmap
p3 = CACHE / 'phase3'
manifest_p3 = p3 / 'rdm_manifest.json'
if manifest_p3.exists():
    m = json.loads(manifest_p3.read_text())
    rdms = m.get('rdms', {})
    print(f'RDMs built: {len(rdms)}')
    print(f"Recommended order: {m.get('recommended_order')}")

    n = len(rdms)
    ncol = 3
    nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.5 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, entry) in zip(axes, rdms.items()):
        rel = entry.get('relative_path')
        if rel:
            with open(p3 / rel, 'rb') as f:
                artifact = pickle.load(f)
            mat = artifact.get('matrix') if isinstance(artifact, dict) else artifact.matrix
            labels = entry.get('unit_labels', [])
            im = ax.imshow(mat, cmap='viridis', vmin=0)
            ax.set_xticks(range(len(labels)))
            ax.set_yticks(range(len(labels)))
            ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
            ax.set_yticklabels(labels, fontsize=7)
            tier = entry.get('recommended_tier', '')
            ax.set_title(f"{name}\n({tier})", fontsize=9)
            plt.colorbar(im, ax=ax, fraction=0.046)
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 5. Phase 4 — ViT training (single fold demo)

Trains the `EEGInformedViTModel` on JIGSAWS video for one fold with the bridge-EEG config. The Phase 3 manifest's selected RDM becomes the RSA target.

For the full 8-fold LOUO sweep, skip ahead to section 6.

In [ ]:
CONFIG = REPO / 'src' / 'configs' / 'bridge_eeg_rdm.yaml'
FOLD = 'fold_1'
OUT = REPO / 'checkpoints' / 'demo_bridge' / FOLD

run([
    PYEXE, SRC / 'training' / 'train_vit_system.py',
    '--config', CONFIG,
    '--data_root', REPO,
    '--task', 'all',
    '--split', FOLD,
    '--output_dir', OUT,
    '--arm', 'PSM2',
])

In [ ]:
# Plot training / validation loss curves from the demo checkpoint
import torch

ckpt_path = OUT / 'best_model.pth'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    train_losses = ckpt.get('train_losses') or ckpt.get('history', {}).get('train_loss')
    val_losses = ckpt.get('val_losses') or ckpt.get('history', {}).get('val_loss')
    if train_losses and val_losses:
        plt.figure(figsize=(6, 3))
        plt.plot(train_losses, label='train')
        plt.plot(val_losses, label='val')
        plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('Demo training curve')
        plt.legend(); plt.tight_layout(); plt.show()
    print('Best val loss:', ckpt.get('best_val_loss'))
    print('Epoch:       ', ckpt.get('epoch'))

## 6. Full 8-fold LOUO + 4-condition ablation (external)

These are long-running (hours to days). Kick them off from a terminal rather than the notebook:

```bash
./run_8fold_louo.sh                # baseline
./run_8fold_louo_brain.sh          # brain_eye mode
pwsh -File run_ablation_study.ps1  # full 4-condition ablation
```

Once any of those finish, the next cell aggregates and visualizes the results.

In [ ]:
# Aggregate whatever results currently exist
for cond_dir in sorted(ANALYSIS.glob('*/louo_results.json')):
    print(f'Already aggregated: {cond_dir.relative_to(REPO)}')

# If you just want a fresh aggregate on eval_results/, run:
# run([PYEXE, PIPELINE / 'aggregate_louo_results.py',
#      '--eval_dir', REPO / 'eval_results',
#      '--output', REPO / 'louo_summary.txt',
#      '--json_output', REPO / 'louo_results.json'])

In [ ]:
# Visualize per-condition mean±std across folds for headline metrics
conditions = [p.parent.name for p in ANALYSIS.glob('*/louo_results.json')]
if not conditions:
    print('No aggregated results yet.')
else:
    metrics_of_interest = ['gesture_accuracy', 'skill_accuracy', 'position_rmse']
    data = {m: {} for m in metrics_of_interest}
    for cond in conditions:
        res = json.loads((ANALYSIS / cond / 'louo_results.json').read_text())
        for m in metrics_of_interest:
            entry = res.get(m) or res.get('overall', {}).get(m)
            if isinstance(entry, dict):
                data[m][cond] = (entry.get('mean'), entry.get('std'))

    fig, axes = plt.subplots(1, len(metrics_of_interest), figsize=(4 * len(metrics_of_interest), 3))
    for ax, m in zip(np.atleast_1d(axes), metrics_of_interest):
        names = list(data[m].keys())
        means = [data[m][c][0] for c in names]
        stds = [data[m][c][1] for c in names]
        if any(v is None for v in means):
            ax.set_title(f'{m} — missing'); ax.axis('off'); continue
        xs = np.arange(len(names))
        ax.bar(xs, means, yerr=stds, capsize=4, color='steelblue', edgecolor='white')
        ax.set_xticks(xs); ax.set_xticklabels(names, rotation=30, ha='right')
        ax.set_title(m)
    plt.tight_layout(); plt.show()

## 7. Manuscript export (optional)

Generates the LaTeX manuscript sources and figures from aggregated ablation outputs.

In [ ]:
# Uncomment to run:
# run([PYEXE, PIPELINE / 'manuscript_writer.py', '--analysis_root', ANALYSIS])

## 8. Phase A — within-subject split types & skill-manifold analysis

Two analyses independent of the brain-bridge machinery:

- **Within-subject splits** (`inter_trial_within_subject`, `intra_trial_half`): diagnostic split families that measure what fraction of LOUO failure is driven by unseen motor style vs intrinsic task difficulty.
- **Post-hoc skill manifold**: do existing `brain_eye` / `bridge_eeg` checkpoints already carry separable representations of Novice / Intermediate / Expert in their embedding space? (Measured across-fold with each fold's own held-out checkpoint.)

In [ ]:
# Inspect the new split families (run pipeline/generate_splits.py first if missing)
splits_dir = REPO / 'data' / 'splits'
for family_suffix, label in [('', 'louo'),
                             ('_inter_trial_within_subject', 'inter_trial_within_subject'),
                             ('_intra_trial_half', 'intra_trial_half')]:
    fpath = splits_dir / f'Knot_Tying_splits{family_suffix}.json'
    if not fpath.exists():
        print(f'[{label}] missing: {fpath}')
        continue
    d = json.loads(fpath.read_text())
    folds = [k for k in d if k.startswith('fold_')]
    f1 = d[folds[0]]
    print(f'[{label}] Knot_Tying fold_1 -> train={len(f1["train"])}, val={len(f1["val"])}, '
          f'test={len(f1["test"])}, test_surgeon={f1.get("test_surgeon")}')
    if 'segment_filter' in f1:
        any_trial = next(iter(f1['segment_filter']['train']))
        print(f'         segment_filter[train][{any_trial}] = {f1["segment_filter"]["train"][any_trial]}')
        print(f'         segment_filter[test][{any_trial}]  = {f1["segment_filter"]["test"][any_trial]}')

In [ ]:
# Post-hoc skill manifold: aggregate embeddings across folds (each fold uses its own
# model's test set, so each sample is evaluated by a model that never trained on it).
#
# Requires checkpoints under checkpoints/<condition>/all/fold_*/best_model.pth.
# Writes analysis/skill_manifold/<condition>_aggregate_{metrics.json, pca.png, dispersion.png}.

CONDITIONS_TO_ANALYZE = ['brain_eye', 'bridge_eeg']
for cond in CONDITIONS_TO_ANALYZE:
    root = REPO / 'checkpoints' / cond / 'all'
    if not root.exists():
        print(f'[{cond}] no checkpoints under {root}, skipping')
        continue
    run([PYEXE, PIPELINE / 'skill_manifold_analysis.py',
         '--aggregate_root', root,
         '--data_root', REPO,
         '--task', 'all',
         '--split_family', 'louo',
         '--device', 'cuda',
         '--batch_size', '4',
         '--stem', cond])

In [ ]:
# Compare skill separability across conditions
sm_dir = ANALYSIS / 'skill_manifold'
rows = []
for mp in sorted(sm_dir.glob('*_aggregate_metrics.json')):
    d = json.loads(mp.read_text())
    ov = d.get('overall') or {}
    rows.append({
        'condition': mp.stem.replace('_aggregate_metrics', ''),
        'n_samples': d.get('n_samples_total'),
        'within_mean': ov.get('within_mean'),
        'between_mean': ov.get('between_mean'),
        'separability_ratio': ov.get('separability_ratio'),
        'silhouette': ov.get('silhouette'),
    })

if rows:
    header = list(rows[0].keys())
    print('  '.join(f'{h:>20}' for h in header))
    for r in rows:
        print('  '.join(f'{str(r[h])[:20]:>20}' for h in header))

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    names = [r['condition'] for r in rows]
    axes[0].bar(names, [r['separability_ratio'] for r in rows],
                color='steelblue', edgecolor='white')
    axes[0].axhline(1.0, color='k', lw=0.5, ls='--')
    axes[0].set_ylabel('separability ratio (>1 = separable)')
    axes[0].set_title('Between-skill / within-skill distance')
    axes[1].bar(names, [r['silhouette'] for r in rows],
                color='indianred', edgecolor='white')
    axes[1].axhline(0.25, color='k', lw=0.5, ls='--')
    axes[1].axhline(0.0, color='gray', lw=0.5)
    axes[1].set_ylabel('silhouette (>0.25 = moderate)')
    axes[1].set_title('Skill cluster silhouette')
    plt.tight_layout(); plt.show()
else:
    print('No skill_manifold results yet. Run the previous cell first.')

In [ ]:
# Render the saved PCA + dispersion figures inline. One row per condition.
from IPython.display import Image, display, Markdown

sm_dir = ANALYSIS / 'skill_manifold'
conditions = sorted({p.name.replace('_aggregate_pca.png', '')
                     for p in sm_dir.glob('*_aggregate_pca.png')})

for cond in conditions:
    pca_png = sm_dir / f'{cond}_aggregate_pca.png'
    disp_png = sm_dir / f'{cond}_aggregate_dispersion.png'
    metrics_json = sm_dir / f'{cond}_aggregate_metrics.json'

    display(Markdown(f'### `{cond}`'))
    if metrics_json.exists():
        ov = (json.loads(metrics_json.read_text()).get('overall') or {})
        display(Markdown(
            f'- n={ov.get("n_samples")} · within={ov.get("within_mean"):.3f} · '
            f'between={ov.get("between_mean"):.3f} · '
            f'ratio={ov.get("separability_ratio"):.3f} · '
            f'silhouette={ov.get("silhouette", float("nan")):.3f}'
        ))
    if pca_png.exists():
        display(Image(filename=str(pca_png)))
    if disp_png.exists():
        display(Image(filename=str(disp_png)))

# Caveat: each fold's test set is one surgeon = one skill class, so the two
# Expert subclusters visible in PCA are surgeons D vs E rather than meaningful
# intra-skill variance. Surgeon identity is a strong confound of "skill"
# metrics at this fold count.